# Content-Based Recommendation System Notebook
In this notebook, we will explore and implement a content-based recommendation system. Content-based recommendation systems suggest items to users based on the characteristics of the items and a profile of the user's preferences.
This approach is particularly useful when we have a lot of information about the items and the users' preferences. We will build a simple content-based recommendation system using Python and the scikit-learn library.

## Table of Contents

1- Introduction
    What is a Content-Based Recommendation System?
    How Does it Work?
    Data Preparation
2- Dataset
    Feature Extraction
    Data Preprocessing
    Building the Content-Based Recommendation System
3- TF-IDF
    Vectorization
    Cosine Similarity
    Recommending Items
4- Evaluation
    Evaluation Metrics
5- Conclusion
## - Summary
## 1. Introduction
 - What is a Content-Based Recommendation System?
    A content-based recommendation system recommends items to users based on the content or characteristics of the items. This type of recommendation system focuses on    understanding the properties of items and learning user preferences from the items they have interacted with in	 the past.
 - How Does it Work?
   The working principle of a content-based recommendation system can be summarized in a few steps:
        1- Feature Extraction: Extract relevant features from the items. For example, in a movie recommendation system, features could include genre, director, actors, and plot keywords.

        2- User Profile: Create a user profile based on their interactions with items. This profile is essentially a summary of the features of items the user has liked or interacted with in the past.

        3- Recommendation: Calculate the similarity between the user profile and each item's features. Items that are most similar to the user profile are recommended.

## 2. Data Preparation
Dataset
- We will use a dataset containing movie information, including titles, genres, and descriptions.

In [33]:
# Import needed modules
import numpy as np
import pandas as pd
import difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import re

In [3]:
# Read data
df = pd.read_csv("E:\\NLP\\movies.csv")

In [4]:
df.shape

(4803, 24)

In [5]:
# printing the first 2 rows of the dataframe
df.head(2)

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski


In [6]:
# Get data information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 4803 non-null   int64  
 1   budget                4803 non-null   int64  
 2   genres                4775 non-null   object 
 3   homepage              1712 non-null   object 
 4   id                    4803 non-null   int64  
 5   keywords              4391 non-null   object 
 6   original_language     4803 non-null   object 
 7   original_title        4803 non-null   object 
 8   overview              4800 non-null   object 
 9   popularity            4803 non-null   float64
 10  production_companies  4803 non-null   object 
 11  production_countries  4803 non-null   object 
 12  release_date          4802 non-null   object 
 13  revenue               4803 non-null   int64  
 14  runtime               4801 non-null   float64
 15  spoken_languages     

In [7]:
# Selecting the relevant features for recommendation
selected_features  = ["genres" , "keywords" , 'overview' , "title"]

df = df[selected_features]

In [8]:
df.shape

(4803, 4)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   genres    4775 non-null   object
 1   keywords  4391 non-null   object
 2   overview  4800 non-null   object
 3   title     4803 non-null   object
dtypes: object(4)
memory usage: 150.2+ KB


In [10]:
df = df.dropna().reset_index(drop=True)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4387 entries, 0 to 4386
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   genres    4387 non-null   object
 1   keywords  4387 non-null   object
 2   overview  4387 non-null   object
 3   title     4387 non-null   object
dtypes: object(4)
memory usage: 137.2+ KB


In [12]:
# combining all the 5 selected features
df["combined"] = df["genres"] + " " + df["keywords"] + " " + df["overview"] + " " + df["title"]

In [13]:
df.head(2)

,genres,keywords,overview,title,combined
0,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,"In the 22nd century, a paraplegic Marine is di...",Avatar,Action Adventure Fantasy Science Fiction cultu...
1,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,"Captain Barbossa, long believed to be dead, ha...",Pirates of the Caribbean: At World's End,Adventure Fantasy Action ocean drug abuse exot...


In [14]:
data = df[["title" , "combined"]]

In [15]:
data.head(2)

,title,combined
0,Avatar,Action Adventure Fantasy Science Fiction cultu...
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action ocean drug abuse exot...


In [34]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [ ]:
def preprocess_text(text):
    # Remove special characters and numbers
    text = re.sub(r"[^a-zA-Z\s]" , " " , text)
    text = text.lower()
    # Convert to lowercase
    tokens = word_tokenize(text)
    # Tokenize and remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

In [18]:
data["clean_text"] = data["combined"].apply(preprocess_text)

C:\Users\Talaat Mostafa\AppData\Local\Temp\ipykernel_12928\212055494.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["clean_text"] = data["combined"].apply(preprocess_text)


In [19]:
# converting the text data to feature vectors
tfidf_vectorizer = TfidfVectorizer(max_features = 5000, ngram_range=(1,2))
tfidf_matrix = tfidf_vectorizer.fit_transform(data["clean_text"])

In [20]:
# getting the similarity scores using cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix)

In [21]:
print(cosine_sim)

[[1.         0.04441165 0.02539097 ... 0.00335349 0.         0.        ]
 [0.04441165 1.         0.02317264 ... 0.0132465  0.02586368 0.00567565]
 [0.02539097 0.02317264 1.         ... 0.00758224 0.0130973  0.        ]
 ...
 [0.00335349 0.0132465  0.00758224 ... 1.         0.         0.        ]
 [0.         0.02586368 0.0130973  ... 0.         1.         0.04971783]
 [0.         0.00567565 0.         ... 0.         0.04971783 1.        ]]


In [22]:
# creating a list with all the movie names given in the dataset
list_of_movies = data["title"].tolist()

In [23]:
# getting the movie name from the user
#movie_name = input(' Enter your favourite movie name : ')
movie_name = "Iron Man"

In [24]:
# finding the close match for the movie name given by the user
find_close_movie = difflib.get_close_matches(movie_name , list_of_movies)
print(find_close_movie)  # printing the closest match

['Iron Man', 'Iron Man 3', 'Iron Man 2']


In [25]:
index_of_movie = data[data.title == find_close_movie[0]].index[0]
index_of_movie

68

In [26]:
# getting a list of similar movies
similarity_score = list(enumerate(cosine_sim[index_of_movie]))
#similarity_score

In [27]:
# sorting the movies based on their similarity score
sort_similar_movies = sorted(similarity_score , key = lambda x:x[1] , reverse = True)
print(sort_similar_movies)

[(68, 1.0), (78, 0.45365107541692595), (31, 0.37711572930814113), (7, 0.26982710977283825), (506, 0.2632566436566824), (99, 0.23546516195064715), (201, 0.2216124554549362), (180, 0.21927322074385389), (64, 0.2085862197576598), (92, 0.20544377159609653), (16, 0.20238909046512488), (172, 0.1942485267235821), (1118, 0.18932121783021066), (230, 0.1860700998940901), (26, 0.18398455990436766), (33, 0.1827148414459646), (3223, 0.16281470338846885), (781, 0.1623227477054796), (75, 0.1581543229983762), (2165, 0.1548972788750477), (1705, 0.14944062372965636), (120, 0.14537002995088827), (124, 0.14338169149318528), (2159, 0.1425058688496826), (46, 0.14184725682652916), (3200, 0.14128170740702417), (935, 0.13395952209067155), (548, 0.13310801740839065), (1827, 0.1305935915721117), (240, 0.12663587357992967), (4022, 0.12146750541388694), (30, 0.1197285352461907), (1055, 0.11630152099632465), (3782, 0.11618096756491872), (2359, 0.11523549569540116), (137, 0.11507920043399306), (416, 0.11461514625987

In [28]:
top_sim = sort_similar_movies[1:6]
(top_sim)

[(78, 0.45365107541692595),
 (31, 0.37711572930814113),
 (7, 0.26982710977283825),
 (506, 0.2632566436566824),
 (99, 0.23546516195064715)]

In [29]:
# print the name of similar movies based on the index
movie_name = "Iron Man"
print("Movies suggested for you : \n") 
    
i = 1
for movie in top_sim:
        index = movie[0]
        title_movie = data[data.index == index]['title'].values[0]
        print(i , '-' , title_movie)
        i+=1   

Movies suggested for you : 

1 - Iron Man 2
2 - Iron Man 3
3 - Avengers: Age of Ultron
4 - X-Men
5 - X-Men: First Class


# Full Recommendation System

In [30]:
def recommend_movies(movie_name , cosine_sim = cosine_sim , df = data):
    titles = data['title'].str.lower().tolist()
    movie_name = movie_name.lower()
    
    if movie_name in titles:
        index_of_movie = data[data.title.str.lower() == movie_name].index[0]
        
    else:
        close_match = difflib.get_close_matches(movie_name , titles)
        if not close_match:
            return "Movie not found. Please check the spelling or try another movie."
        index_of_movie = data[data.title.str.lower() == close_match[0]].index[0]
        
    similarity_score = list(enumerate(cosine_sim[index_of_movie]))
    sort_similar_movies = sorted(similarity_score , key = lambda x : x[1] , reverse = True)
    
    top_movies = sort_similar_movies[1:6]
    
    print('Movies suggested for you : \n')
    i = 1
    
    for movie in top_movies:
        index = movie[0]
        title_movie = data[data.index == index]['title'].values[0]
        print(i , '-' , title_movie)
        i+=1  

In [32]:
movie_name = "Avatar"
print(f"Recommendations for the Movie")
recommendations = recommend_movies(movie_name)

Recommendations for the Movie
Movies suggested for you : 

1 - Lost in Space
2 - Lifeforce
3 - Moonraker
4 - Zathura: A Space Adventure
5 - Space Chimps
